In [25]:
## nba_api try

## Notes it only works from 1983 on.

from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd

In [26]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import time

def get_all_nba_schedules(start_year=1983, end_year=2025):
    """
    Get all NBA regular season schedules from start_year to end_year.
    
    Parameters:
    - start_year: First year to fetch (default: 1983)
    - end_year: Last year to fetch (default: 2025)
    
    Returns:
    - DataFrame with all games
    """
    all_games = []
    
    for year in range(start_year, end_year):
        # NBA seasons are formatted as 'YYYY-YY' (e.g., '1983-84')
        season = f"{year}-{str(year + 1)[-2:]}"
        
        print(f"Fetching season {season}...")
        
        try:
            # Get games for this season
            gamefinder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                league_id_nullable='00',  # '00' = NBA
                season_type_nullable='Regular Season'
            )
            
            games = gamefinder.get_data_frames()[0]
            all_games.append(games)
            
            print(f"  Found {len(games)} game records for {season}")
            
            # Be respectful to the API - add a small delay
            time.sleep(0.6)
            
        except Exception as e:
            print(f"  Error fetching {season}: {e}")
            continue
    
    # Combine all seasons into one DataFrame
    if all_games:
        full_schedule = pd.concat(all_games, ignore_index=True)
        print(f"\nTotal game records: {len(full_schedule)}")
        return full_schedule
    else:
        print("No games found")
        return pd.DataFrame()

In [27]:
schedules = get_all_nba_schedules(1983, 2025)

Fetching season 1983-84...
  Found 1886 game records for 1983-84
Fetching season 1984-85...
  Found 1886 game records for 1984-85
Fetching season 1985-86...
  Found 1886 game records for 1985-86
Fetching season 1986-87...
  Found 1886 game records for 1986-87
Fetching season 1987-88...
  Found 1886 game records for 1987-88
Fetching season 1988-89...
  Found 2050 game records for 1988-89
Fetching season 1989-90...
  Found 2214 game records for 1989-90
Fetching season 1990-91...
  Found 2214 game records for 1990-91
Fetching season 1991-92...
  Found 2214 game records for 1991-92
Fetching season 1992-93...
  Found 2214 game records for 1992-93
Fetching season 1993-94...
  Found 2214 game records for 1993-94
Fetching season 1994-95...
  Found 2214 game records for 1994-95
Fetching season 1995-96...
  Found 2378 game records for 1995-96
Fetching season 1996-97...
  Found 2378 game records for 1996-97
Fetching season 1997-98...
  Found 2378 game records for 1997-98
Fetching season 1998-99..

C:\Users\ryanj\AppData\Local\Temp\ipykernel_35328\2578252527.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_schedule = pd.concat(all_games, ignore_index=True)


In [28]:
all_teams = set(schedules['TEAM_ABBREVIATION'])

season_ids = set(schedules['SEASON_ID'])

franchise_map = {
    # Utah Jazz
    "UTA": "Utah Jazz",
    "UTH": "Utah Jazz",

    # Golden State Warriors
    "GSW": "Golden State Warriors",
    "GOS": "Golden State Warriors",

    # San Antonio Spurs
    "SAS": "San Antonio Spurs",
    "SAN": "San Antonio Spurs",

    # Philadelphia 76ers
    "PHI": "Philadelphia 76ers",
    "PHL": "Philadelphia 76ers",

    # Sacramento / Kansas City Kings
    "SAC": "Sacramento Kings",
    "KCK": "Sacramento Kings",

    # Vancouver / Memphis Grizzlies
    "VAN": "Memphis Grizzlies",
    "MEM": "Memphis Grizzlies",

    # New Jersey / Brooklyn Nets
    "NJN": "Brooklyn Nets",
    "BKN": "Brooklyn Nets",

    # San Diego / LA Clippers
    "SDC": "Los Angeles Clippers",
    "LAC": "Los Angeles Clippers",

    # New Orleans franchise (Hornets → Pelicans)
    "NOH": "New Orleans Pelicans",
    "NOK": "New Orleans Pelicans",
    "NOP": "New Orleans Pelicans",

    # Charlotte franchise (Bobcats → Hornets)
    "CHH": "Charlotte Hornets",
    "CHA": "Charlotte Hornets",

    # Single-abbreviation franchises
    "LAL": "Los Angeles Lakers",
    "BOS": "Boston Celtics",
    "CHI": "Chicago Bulls",
    "NYK": "New York Knicks",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "CLE": "Cleveland Cavaliers",
    "DET": "Detroit Pistons",
    "IND": "Indiana Pacers",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "ATL": "Atlanta Hawks",
    "MIA": "Miami Heat",
    "ORL": "Orlando Magic",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "TOR": "Toronto Raptors",
    "OKC": "Oklahoma City Thunder",
    "SEA": "Oklahoma City Thunder",  # SuperSonics history stays with Thunder
    "WAS": "Washington Wizards",
    "HOU": "Houston Rockets"
}

In [29]:
def generateDataFrame(schedules, season_ids, franchise_map):
    """
    Build a DataFrame where each row corresponds to a (season, franchise)
    pair and contains the full win/loss sequence for that team-season.

    Parameters
    ----------
    schedules : pd.DataFrame
        Game-level schedule data containing at least:
        - TEAM_ABBREVIATION
        - SEASON_ID
        - WL (win/loss indicator)

    season_ids : iterable
        Collection of season identifiers (e.g. 22017, 22018, ...)

    franchise_map : dict
        Mapping from team abbreviations to franchise identifiers

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - League
        - Season
        - Team (franchise)
        - Sequence (list of binary win/loss outcomes)
    """

    rows = []

    for season in season_ids:
        for team in all_teams:
            temp = schedules[(schedules['TEAM_ABBREVIATION'] == team) & (schedules['SEASON_ID'] == season)]
            if temp.empty:
                continue

            wl = list(temp["WL"])
            binary_sequence = [1 if x == "W" else 0 for x in wl]

            rows.append({
                "League" : "NBA",
                "Season" : str(int(season) - 20000), ## Conversion from schedule season id. 
                "Team" : franchise_map[team],
                "Sequence" : binary_sequence,
            })

    out = pd.DataFrame(rows)

    return out

In [30]:
## Writing out to processed data directory

output = "../data/processed/nba.csv"

main = generateDataFrame(schedules, season_ids, franchise_map)

main.to_csv(output)